In [ ]:
#IMPORTS:

import torch
import matplotlib.pyplot as plt

from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer
)

from xray.interpretability import DistilBertInterpreter

: 

In [ ]:
# LOAD THE HUGGINGFACE MODEL:

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
).to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
)

print("Model loaded!")

In [ ]:
# CREATE THE INTERPRETER:

interpreter = DistilBertInterpreter(
    model,
    tokenizer,
    device
)

In [ ]:
# EXPLAIN A SENTENCE:

text = "This movie was absolutely fantastic and beautifully acted."

(
    tokens,
    attributions,
    delta,
    input_output,
    baseline_output,
    total_attribution,
    completeness_error
) = interpreter.attribute(text)

words, scores = interpreter.aggregate_tokens(
    tokens,
    attributions
)

print("Word-level attributions:")

for word, score in zip(words, scores):
    print(f"{word:15} {float(score):+.4f}")

print("\nCompleteness error:", float(completeness_error))

In [ ]:
# ADD PREDICTION INFORMATION:

encoding = tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

encoding = {
    key: value.to(device)
    for key, value in encoding.items()
}

with torch.no_grad():
    outputs = model(**encoding)

probabilities = torch.softmax(
    outputs.logits,
    dim=1
)

prediction = torch.argmax(
    probabilities,
    dim=1
).item()

confidence = probabilities[0, prediction].item()

label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}[prediction]

print("Prediction:", label)
print("Confidence:", f"{confidence:.4f}")